# Process RNAseq and ATAC modalities for MOFA
This notebook, currently, focuses on 3 timepoints, pre, acute and post COVID, samples of the 1st individual from:

Wimmers F, Burrell AR, Feng Y, Zheng H et al. Multi-omics analysis of mucosal and systemic immunity to SARS-CoV-2 after birth. Cell 2023 Oct 12;186(21):4632-4651.e23. PMID: 37776858


Loads RNA-SEQ and ATAC-seq data, performs QC and normalizitaion steps followed by re-naming "var"iables to match between modalities.

Then MOFA is used to identify major sources of variation across RNA-SEQ and ATAC-seq data.

TODO: combine all samples

In [17]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from pathlib import Path
import muon as mu
import snapatac2 as snap
import hdf5plugin
from muon import atac as ac
import logging
logger = logging.getLogger(__name__)

In [ ]:
# This fetches the "Friendly Name" (external_gene_name) for your Ensembl IDs
gene_names = sc.queries.biomart_annotations(
    "hsapiens", 
    ["ensembl_gene_id", "external_gene_name"]
)

In [86]:
mapping_dict = (
    gene_names.dropna(subset=['external_gene_name'])
    .set_index('ensembl_gene_id')['external_gene_name']
    .to_dict()
)

In [3]:
sample_dir = Path("./data/")
rna_dirs = [d for d in sample_dir.glob("*_gex_*") if d.is_dir()]
atac_dirs = [d for d in sample_dir.glob("*_atac_*") if d.is_dir()]
rna_ind_h5ads_dir = Path("./data/rna_samples_h5ad" )
atac_ind_h5ads_dir = Path("./data/atac_samples_h5ad")

In [153]:
# Process RNAseq
for sample in rna_dirs[0:3]:
    parts = sample.name.split("_")
    gsm, ind_id = parts[1], "_".join([parts[3], parts[4]])
    prefix = "_".join([gsm, ind_id]) + "."
    rna = sc.read_10x_mtx(sample, prefix = prefix)
    # filter high mitochondrial content cells
    rna.var['mt'] = rna.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
    sc.pp.calculate_qc_metrics(rna, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    # filter genes with no expression
    mu.pp.filter_var(rna, 'n_cells_by_counts', lambda x: x >= 3)
    
    # filter cells with no expression
    mu.pp.filter_obs(rna, 'n_genes_by_counts', lambda x: (x >= 200) & (x < 5000))
    mu.pp.filter_obs(rna, 'total_counts', lambda x: x < 15000)
    mu.pp.filter_obs(rna, 'pct_counts_mt', lambda x: x < 20)
    
    # normalize
    sc.pp.normalize_total(rna, target_sum=1e4)
    sc.pp.log1p(rna)
    
    # label highly variable genes
    sc.pp.highly_variable_genes(rna, min_mean=0.02, max_mean=4, min_disp=0.5)
    rna = rna[:, rna.var.highly_variable].copy()
    # save in raw
    #rna.raw = rna
    # scale counts to zero
    sc.pp.scale(rna, max_value=10)

    # add pca
    #sc.tl.pca(rna, svd_solver='arpack')
    pathToWrite = (rna_ind_h5ads_dir / ind_id).with_suffix(".h5ad")
    rna.write_h5ad(pathToWrite)

Only considering the two last: ['.mtx', '.gz'].
Only considering the two last: ['.mtx', '.gz'].


... storing 'feature_types' as categorical


Only considering the two last: ['.mtx', '.gz'].
Only considering the two last: ['.mtx', '.gz'].


... storing 'feature_types' as categorical


Only considering the two last: ['.mtx', '.gz'].
Only considering the two last: ['.mtx', '.gz'].


... storing 'feature_types' as categorical


In [149]:
# Process ATACseq
for sample in atac_dirs[0:3]:
    parts = sample.name.split("_")
    gsm, ind_id = parts[1], "_".join([parts[3], parts[4]])
    prefix = "_".join([gsm, ind_id]) + "."
    fragments_tsv = list(sample.rglob("*.tsv.gz"))
    logger.info(f"Sample = {ind_id}")
    
    # output dir h5ad
    pathToWrite = (atac_ind_h5ads_dir / ind_id).with_suffix(".h5ad")
    
    atac = snap.pp.import_fragments(fragments_tsv[0],
                                    chrom_sizes=snap.genome.hg38,
                                    #file = pathToWrite = (rna_ind_h5ads_dir / ind_id).with_suffix(".h5ad"),
                                    sorted_by_barcode=False)
    # QC
    snap.metrics.tsse(atac, snap.genome.hg38)
    snap.pp.filter_cells(atac, min_tsse=7)
    #snap.pp.add_tile_matrix(atac, bin_size=50000, n_jobs=-1)
    atac = snap.pp.make_gene_matrix(atac, gene_anno=snap.genome.hg38)
    sc.pp.calculate_qc_metrics(atac, percent_top=None, log1p=False, inplace=True)
    #snap.pp.add_tile_matrix(atac, bin_size=50000, n_jobs=-1)
    #snap.pp.select_features(atac, n_features=50000)
    #snap.pp.scrublet(atac)
    #snap.pp.filter_doublets(atac)
    #break
    # filter peaks which expression is not detected
    mu.pp.filter_var(atac, 'n_cells_by_counts', lambda x: x >= 10)

    # filter cells with no expression
    mu.pp.filter_obs(atac, 'n_genes_by_counts', lambda x: (x >= 1000) & (x <= 15000))
    mu.pp.filter_obs(atac, 'total_counts', lambda x: (x >= 4000) & (x <= 40000))

    # atac specific qc
    #atac.obs['NS']=1
    #ac.tl.nucleosome_signal(atac, n=1e6)

    # normalize
    sc.pp.normalize_total(atac, target_sum=1e4)
    sc.pp.log1p(atac)

    # variable peaks/genes
    sc.pp.highly_variable_genes(atac, min_mean=0.05, max_mean=1.5, min_disp=.5)
    atac.raw = atac
    atac = atac[:, atac.var.highly_variable].copy()

    # latent components
    ac.tl.lsi(atac)
    atac.obsm['X_lsi'] = atac.obsm['X_lsi'][:,1:]
    atac.varm["LSI"] = atac.varm["LSI"][:,1:]
    atac.uns["lsi"]["stdev"] = atac.uns["lsi"]["stdev"][1:]

    # generate neighbors
    sc.pp.neighbors(atac, use_rep="X_lsi", n_neighbors=10, n_pcs=30)
    # add pca
    #sc.tl.pca(rna, svd_solver='arpack')

    # non-scaled data to raw
    # atac.raw = atac
    # Scale for MOFA (Centering and scaling is highly recommended for MOFA)
    sc.pp.scale(atac, max_value=10)

    # fix var names
    mapping_dict = gene_names.dropna(subset=['external_gene_name', 'ensembl_gene_id']).set_index('external_gene_name')['ensembl_gene_id'].to_dict()# strip version dots
    current_names = atac.var_names.tolist()
    new_names = []
    for name in current_names:
        clean_name = name.split('.')[0]
        new_id = mapping_dict.get(clean_name, clean_name)
        new_names.append(new_id)

    atac.var_names = new_names
    atac.var_names_make_unique()
    # save
    atac.write_h5ad(pathToWrite)

2026-03-20 11:27:46 - INFO - Sample = 30_CC0022
2026-03-20 11:33:51 - INFO - Sample = 31_CC0022
2026-03-20 11:38:53 - INFO - Sample = 32_CC0022


In [154]:
# MOFA per ind
for sample in atac_dirs[0:3]:
    parts = sample.name.split("_")
    gsm, ind_id = parts[1], "_".join([parts[3], parts[4]])
    prefix = "_".join([gsm, ind_id]) + "."
    logger.info(f"Sample = {ind_id}")
    # rna h5ad
    pathRNA = (rna_ind_h5ads_dir / ind_id).with_suffix(".h5ad")
    # atac h5ad
    pathATAC = (atac_ind_h5ads_dir / ind_id).with_suffix(".h5ad")
    rna = sc.read_h5ad(pathRNA)
    atac = sc.read_h5ad(pathATAC)

    # fix rna
    rna.var["gene_name"] = rna.var.gene_ids
    rna.var['gene_name'] = rna.var['gene_name'].replace(mapping_dict)
    rna.var_names = rna.var['gene_name']

    mdata = mu.MuData({'rna': rna, 'atac': atac})
    mu.pp.intersect_obs(mdata)

    for modality in mdata.mod.keys():
    # Ensure the data is at least float32
        if not np.issubdtype(mdata.mod[modality].X.dtype, np.floating):
            print(f"Casting {modality} to float32...")
            mdata.mod[modality].X = mdata.mod[modality].X.astype(np.float32)
    
    mu.tl.mofa(mdata, outfile=(Path("data")/ind_id).with_suffix(".hdf5"), gpu_mode = True)


2026-03-20 12:41:24 - INFO - Sample = 30_CC0022



        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Loaded view='rna' group='group1' with N=6988 samples and D=3934 features...
Loaded view='atac' group='group1' with N=6988 samples and D=5138 features...


Model options:
- Automatic Relevance Determination prior on the factors: True
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Lik

2026-03-20 12:43:04 - INFO - Sample = 31_CC0022


Saved MOFA embeddings in .obsm['X_mofa'] slot and their loadings in .varm['LFs'].

        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Loaded view='rna' group='group1' with N=6472 samples and D=3566 features...
Loaded view='atac' group='group1' with N=6472 samples and D=3713 features...


Model options:
- Automatic Relevance Determination prior on the factors: True
- Automatic Relevance Determination prior on the weights: True
- Spike-an

2026-03-20 12:44:17 - INFO - Sample = 32_CC0022


Saved MOFA embeddings in .obsm['X_mofa'] slot and their loadings in .varm['LFs'].

        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Loaded view='rna' group='group1' with N=5497 samples and D=3636 features...
Loaded view='atac' group='group1' with N=5497 samples and D=4160 features...


Model options:
- Automatic Relevance Determination prior on the factors: True
- Automatic Relevance Determination prior on the weights: True
- Spike-an